# ✅ Solution 5: Studi Kasus Harga Properti

**Approach**: Multi-wilayah analysis, adaptive metric selection, storytelling visualization  
**Key Insight**: Harga properti log-normal → selalu right-skewed → median adalah standar industri real estate.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import trim_mean

np.random.seed(42)

def generate_harga(mean, std, n, outliers=[]):
    data = np.random.lognormal(np.log(mean), std, n)
    return np.concatenate([data, np.array(outliers)])

harga_jaksel = generate_harga(3.5, 0.6, 120, [25, 35, 50, 75])
harga_jakpus = generate_harga(5.0, 0.7, 80, [40, 60, 80, 100])
harga_jakbar = generate_harga(2.0, 0.5, 100, [15, 20])
harga_jaktim = generate_harga(1.5, 0.4, 130, [10, 12])
harga_jakut = generate_harga(2.5, 0.6, 90, [18, 25, 30])

df_properti = pd.DataFrame({
    'harga_miliar': np.concatenate([harga_jaksel, harga_jakpus, harga_jakbar, harga_jaktim, harga_jakut]),
    'wilayah': (
        ['Jakarta Selatan'] * len(harga_jaksel) + ['Jakarta Pusat'] * len(harga_jakpus) +
        ['Jakarta Barat'] * len(harga_jakbar) + ['Jakarta Timur'] * len(harga_jaktim) +
        ['Jakarta Utara'] * len(harga_jakut)
    )
})

In [ ]:
# SOLUSI TODO 1: Analisis per wilayah
def wilayah_stats(group):
    return pd.Series({
        'n_properti': len(group),
        'mean': group.mean(),
        'median': group.median(),
        'trimmed_10pct': trim_mean(group.values, 0.1),
        'min': group.min(),
        'max': group.max(),
        'skewness': group.skew(),
    })

df_stats = df_properti.groupby('wilayah')['harga_miliar'].apply(wilayah_stats).unstack()
df_stats['gap_persen'] = ((df_stats['mean'] - df_stats['median']) / df_stats['median'] * 100)
df_stats = df_stats.sort_values('median', ascending=False)

print('Statistik Harga Properti per Wilayah (dalam miliar Rupiah):')
display(df_stats.round(2))

In [ ]:
# SOLUSI TODO 2: Harga tipikal per wilayah
def pilih_harga_tipikal(row):
    if row['skewness'] > 1.5:
        return row['median'], 'Median (skewness > 1.5)'
    elif row['skewness'] > 0.5:
        return row['trimmed_10pct'], 'Trimmed Mean 10% (0.5 < skew ≤ 1.5)'
    else:
        return row['mean'], 'Mean (skewness ≤ 0.5)'

df_stats['harga_tipikal'] = df_stats.apply(lambda r: pilih_harga_tipikal(r)[0], axis=1)
df_stats['alasan_pilihan'] = df_stats.apply(lambda r: pilih_harga_tipikal(r)[1], axis=1)

print('Harga "Tipikal" per Wilayah:')
print('=' * 70)
for wil in df_stats.index:
    row = df_stats.loc[wil]
    print(f'  {wil:18s}: Rp {row["harga_tipikal"]:.2f} Miliar')
    print(f'    Metode: {row["alasan_pilihan"]}')
    print(f'    (Mean={row["mean"]:.2f}, Median={row["median"]:.2f}, Skew={row["skewness"]:.2f})')
    print()

In [ ]:
# SOLUSI TODO 3: Visualisasi storytelling
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# [0,0] Boxplot horizontal
wilayah_order = df_stats.index.tolist()  # sorted by median
sns.boxplot(data=df_properti, y='wilayah', x='harga_miliar', 
            order=wilayah_order, ax=axes[0,0], palette='RdYlGn_r')
axes[0,0].set_title('Distribusi Harga Properti per Wilayah Jakarta', fontweight='bold')
axes[0,0].set_xlabel('Harga (Miliar Rp)')
axes[0,0].set_ylabel('')

# [0,1] Bar chart: mean vs median vs harga_tipikal
x = range(len(wilayah_order))
w = 0.25
axes[0,1].barh([i-w for i in x], [df_stats.loc[wil, 'mean'] for wil in wilayah_order],
              w, label='Mean', color='red', alpha=0.7)
axes[0,1].barh(x, [df_stats.loc[wil, 'median'] for wil in wilayah_order],
              w, label='Median', color='green', alpha=0.7)
axes[0,1].barh([i+w for i in x], [df_stats.loc[wil, 'harga_tipikal'] for wil in wilayah_order],
              w, label='Harga Tipikal', color='blue', alpha=0.7)
axes[0,1].set_yticks(x)
axes[0,1].set_yticklabels(wilayah_order, fontsize=9)
axes[0,1].set_title('Mean vs Median vs Harga Tipikal', fontweight='bold')
axes[0,1].set_xlabel('Harga (Miliar Rp)')
axes[0,1].legend()

# [1,0] Histogram keseluruhan
all_harga = df_properti['harga_miliar']
axes[1,0].hist(all_harga[all_harga < 20], bins=40, color='#4CAF50', edgecolor='black', alpha=0.7)
axes[1,0].axvline(all_harga.mean(), color='red', linestyle='--', lw=2, label=f'Mean = {all_harga.mean():.1f}M')
axes[1,0].axvline(all_harga.median(), color='blue', linestyle='-', lw=2, label=f'Median = {all_harga.median():.1f}M')
axes[1,0].set_title('Distribusi Harga Seluruh Jakarta\n(filter < 20M untuk visibility)', fontweight='bold')
axes[1,0].set_xlabel('Harga (Miliar Rp)')
axes[1,0].set_ylabel('Jumlah Properti')
axes[1,0].legend()

# [1,1] Scatter: skewness vs gap
for wil in wilayah_order:
    row = df_stats.loc[wil]
    axes[1,1].scatter(row['skewness'], row['gap_persen'], s=row['n_properti']*3, alpha=0.7)
    axes[1,1].annotate(wil.replace('Jakarta ', 'Jak'), 
                       (row['skewness'], row['gap_persen']),
                       textcoords='offset points', xytext=(5, 5), fontsize=9)
axes[1,1].axhline(0, color='gray', linestyle=':', alpha=0.5)
axes[1,1].axvline(1.5, color='red', linestyle='--', alpha=0.5, label='Threshold skew=1.5')
axes[1,1].set_xlabel('Skewness')
axes[1,1].set_ylabel('Mean-Median Gap (%)')
axes[1,1].set_title('Skewness vs Gap\n(ukuran dot = jumlah properti)', fontweight='bold')
axes[1,1].legend()

plt.suptitle('Berapa Harga Rumah di Jakarta Sebenarnya?', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# SOLUSI TODO 4: Kesimpulan untuk artikel
overall_median = df_properti['harga_miliar'].median()
overall_mean = df_properti['harga_miliar'].mean()

print('=' * 70)
print('📰 KESIMPULAN UNTUK ARTIKEL')
print('   "Berapa Harga Rumah di Jakarta Sebenarnya?"')
print('=' * 70)
print()
print(f'1. HARGA RUMAH TIPIKAL DI JAKARTA:')
print(f'   Rp {overall_median:.1f} Miliar (median dari {len(df_properti)} properti)')
print(f'   Bukan Rp {overall_mean:.1f} Miliar (mean) — yang diinflasi oleh')
print(f'   properti super mewah.')
print()
print(f'2. PER WILAYAH (dari termahal ke termurah):')
for i, wil in enumerate(wilayah_order, 1):
    row = df_stats.loc[wil]
    print(f'   {i}. {wil:18s}: Rp {row["harga_tipikal"]:.1f} Miliar')
print()
print(f'3. MENGAPA KAMI PAKAI MEDIAN?')
print(f'   Distribusi harga properti selalu "miring ke kanan" karena')
print(f'   ada rumah-rumah mewah (> Rp 20 Miliar) yang menarik rata-rata naik.')
print(f'   Mean harga = Rp {overall_mean:.1f}M, tapi {(df_properti["harga_miliar"] < overall_mean).mean()*100:.0f}% properti')
print(f'   harganya DI BAWAH angka itu. Median lebih mewakili harga yang')
print(f'   akan ditemui pembeli rumah pada umumnya.')
print()
print(f'4. PERINGATAN:')
print(f'   Harga tertinggi dalam data: Rp {df_properti["harga_miliar"].max():.1f} Miliar')
print(f'   Properti premium ini menarik "rata-rata" naik signifikan.')

## 📌 Takeaways

- Harga properti **selalu right-skewed** di seluruh dunia (karena batas bawah tapi tanpa batas atas)
- **Median** adalah standar industri real estate untuk melaporkan harga (bukan mean)
- Setiap wilayah punya skewness berbeda → perlu metric yang adaptive
- Visualisasi boxplot/violin sangat efektif untuk audiens non-teknis
- Alternative approach: Log-transform harga → distribusi jadi normal → mean dari log-harga